# Demo Clase — Agente IA Triage (OpenAI Agents SDK)

Este notebook es solo para **demo de métricas** en clase.
El agente real vive en `src/agent.py`. Para interacción: `python cli.py`.

## 1. Qué es un Agente (vs simple LLM)

| Concepto | LLM simple | Agente real |
|---------|-----------|-------------|
| Entrada → salida | sí | sí |
| Tool use (APIs, DB) | no | **sí** |
| Loop autónomo de decisión | no | **sí** |
| Guardrails | manual | **nativos** |
| Memoria / contexto | limitado | **sí** |

Nuestro agente usa **OpenAI Agents SDK** (oficial, marzo 2025).

In [ ]:
import asyncio, json, time
from pathlib import Path
from src.agent import run_triage, calcular_costo, MODEL
print(f'Modelo: {MODEL}')

## 2. Demo: 1 caso

In [ ]:
sintomas = 'Hombre 60 años, dolor opresivo en pecho con sudoración, irradia al brazo izquierdo, hace 15 min. Vive en San Isidro.'

t0 = time.time()
resp = await run_triage(sintomas)
lat = time.time() - t0

print(json.dumps(resp.result.model_dump(), indent=2, ensure_ascii=False))
print(f'\n--- FinOps ---')
print(f'Requests al LLM: {resp.total_usage["requests"]}')
print(f'Tokens: {resp.total_usage["input_tokens"]} in + {resp.total_usage["output_tokens"]} out')
print(f'Costo: ${calcular_costo(resp.total_usage):.6f}')
print(f'Latencia: {lat:.2f}s')

## 3. Guardrail: el agente rechaza pedidos fuera de alcance

In [ ]:
resp = await run_triage('Qué medicamento me recomiendas para la gripe? Dame la dosis.')
print(f'Bloqueado: {resp.bloqueado_por_guardrail}')
print(f'Motivo: {resp.motivo_bloqueo}')

## 4. Suite completa con métricas FinOps

In [ ]:
from tabulate import tabulate

casos = json.loads(Path('data/casos_prueba.json').read_text(encoding='utf-8'))
rows, aciertos, costo_total = [], 0, 0.0

for c in casos:
    t0 = time.time()
    resp = await run_triage(c['descripcion'])
    lat = time.time() - t0
    if resp.bloqueado_por_guardrail:
        rows.append([c['id'], c['esperado'], 'BLOQUEADO', '-', '-', f'{lat:.2f}', '✗'])
        continue
    r = resp.result
    costo = calcular_costo(resp.total_usage)
    costo_total += costo
    ok = r.nivel.value == c['esperado']
    aciertos += int(ok)
    rows.append([
        c['id'], c['esperado'], r.nivel.value,
        f"{r.confianza:.2f}",
        f"{resp.total_usage['input_tokens']}+{resp.total_usage['output_tokens']}",
        f'{costo:.6f}', f'{lat:.2f}', '✓' if ok else '✗'
    ])

print(tabulate(rows, headers=['ID','Esperado','Predicho','Conf','Tokens','Costo$','Lat s','OK']))
print(f'\nAccuracy: {aciertos}/{len(casos)} = {aciertos/len(casos)*100:.1f}%')
print(f'Costo total: ${costo_total:.6f}')
print(f'Costo promedio: ${costo_total/len(casos):.6f}/caso')

## 5. Proyección de negocio

In [ ]:
costo_prom = costo_total / len(casos)
precio_venta = 0.05  # USD por triage

for pacientes_dia in [100, 1000, 10000, 100000]:
    costo_mes = costo_prom * pacientes_dia * 30
    ingreso_mes = precio_venta * pacientes_dia * 30
    print(f'{pacientes_dia:>7}/día → costo ${costo_mes:>8.2f}/mes · ingreso ${ingreso_mes:>10.2f}/mes · margen {(1 - costo_mes/ingreso_mes)*100:.1f}%')

## Cierre

- **Agente real** (no LLM wrapper): tool use + guardrails + structured output
- **FinOps**: centavos por triage
- **Musk**: 1 archivo de agente, 1 de tools, 1 de schemas. Cero sobre-ingeniería.
- **Zero-hallucination**: Pydantic schema + guardrails + disclaimers + escalamiento humano